In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from glob import glob
from scipy.interpolate import interp1d
from scipy.optimize import brentq

plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 16,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "lines.linewidth": 2,
    "lines.markersize": 6,
    "figure.figsize": (5, 4),
})


In [ ]:
# ==========================================
# 1. Helpers for postprocessed AN/node/M spectra
# ==========================================

base_dir = "./"

def discover_temperatures(base_dir="./"):
    summary = os.path.join(base_dir, "summary_all.csv")
    if os.path.exists(summary):
        df = pd.read_csv(summary)
        return np.sort(df['T'].dropna().unique())
    vals = []
    for path in glob(os.path.join(base_dir, "T_*")):
        try:
            vals.append(float(os.path.basename(path).split("_", 1)[1]))
        except ValueError:
            pass
    return np.array(sorted(vals))

def find_T_dir(T, base_dir="./"):
    candidates = [f"T_{T}", f"T_{T:g}", f"T_{T:.3f}", f"T_{T:.2f}"]
    for name in candidates:
        path = os.path.join(base_dir, name)
        if os.path.isdir(path):
            return path
    return os.path.join(base_dir, f"T_{T:g}")

def read_omega0_series(data_dir, filename, value_col, err_col='Error'):
    file_path = os.path.join(data_dir, filename)
    if not os.path.exists(file_path):
        print(f"Warning: File not found {file_path}")
        return np.nan, np.nan, np.nan
    df = pd.read_csv(file_path).sort_values('omega')
    idx0 = int(np.argmin(np.abs(df['omega'].to_numpy())))
    err = df[err_col].iloc[idx0] if err_col in df.columns else np.nan
    return df[value_col].iloc[idx0], err, df['omega'].iloc[idx0]

def read_peak_summary(data_dir):
    path = os.path.join(data_dir, 'spectra_path_peaks.csv')
    if not os.path.exists(path):
        return pd.DataFrame()
    return pd.read_csv(path)


In [ ]:
# ==========================================
# 2. Main loop
# ==========================================

T_list = discover_temperatures(base_dir)
T_list = T_list[(T_list >= 0.005) & (T_list <= 0.100)]

results = {
    "T": [],
    "AN0": [], "AN0_err": [],
    "node0": [], "node0_err": [],
    "M0": [], "M0_err": [],
    "AN_kx": [], "AN_ky": [],
    "node_kx": [], "node_ky": [],
}

for T in T_list:
    data_dir = find_T_dir(T, base_dir)
    print(f"Processing {os.path.basename(data_dir)}...", end='\r')
    an0, an_err, _ = read_omega0_series(data_dir, 'spectra_dos_AN.csv', 'DOS_AN')
    nd0, nd_err, _ = read_omega0_series(data_dir, 'spectra_dos_node.csv', 'DOS_node')
    m0, m_err, _ = read_omega0_series(data_dir, 'spectra_dos_M.csv', 'DOS_M')
    peaks = read_peak_summary(data_dir)

    results['T'].append(T)
    results['AN0'].append(an0); results['AN0_err'].append(an_err)
    results['node0'].append(nd0); results['node0_err'].append(nd_err)
    results['M0'].append(m0); results['M0_err'].append(m_err)

    for kind, prefix in [('AN', 'AN'), ('node', 'node')]:
        sub = peaks[peaks['kind'] == kind] if not peaks.empty and 'kind' in peaks.columns else pd.DataFrame()
        if len(sub) > 0:
            results[f'{prefix}_kx'].append(sub['kx'].mean())
            results[f'{prefix}_ky'].append(sub['ky'].mean())
        else:
            results[f'{prefix}_kx'].append(np.nan)
            results[f'{prefix}_ky'].append(np.nan)

print()
print("Analysis complete.")
for key in results:
    results[key] = np.array(results[key])


In [ ]:
# ==========================================
# 3. A(omega) at postprocessed AN/node for different T
# ==========================================

def read_series(data_dir, filename, value_col):
    file_path = os.path.join(data_dir, filename)
    if not os.path.exists(file_path):
        return None
    df = pd.read_csv(file_path).sort_values('omega')
    err = df['Error'].to_numpy() if 'Error' in df.columns else None
    return df['omega'].to_numpy(), df[value_col].to_numpy(), err

fig, ax = plt.subplots(dpi=300)
colors = plt.cm.viridis(np.linspace(0, 1, len(T_list)))

for i, T in enumerate(T_list):
    data_dir = find_T_dir(T, base_dir)
    out = read_series(data_dir, 'spectra_dos_AN.csv', 'DOS_AN')
    if out is None:
        continue
    omega, vals, err = out
    ax.plot(omega, vals, color=colors[i], label=rf"T={T:g}")
    if err is not None:
        mask = np.isfinite(vals) & np.isfinite(err)
        ax.fill_between(omega[mask], vals[mask] - err[mask], vals[mask] + err[mask],
                        color=colors[i], alpha=0.15)

ax.set_xlabel(r'$\omega$')
ax.set_ylabel(r'$A_{\mathrm{AN}}(\omega)$')
ax.set_xlim(-0.3, 0.3)
ax.legend(frameon=False, ncol=2)
plt.show()


In [ ]:
df = pd.read_csv("summary_all.csv").sort_values('T')

rho_col = 'Superfluid_Stiffness_mean'
Tc = np.nan
if rho_col in df.columns and len(df) >= 2:
    diff = df[rho_col].to_numpy() - (2 / np.pi) * df['T'].to_numpy()
    if np.any(np.isfinite(diff[:-1]) & np.isfinite(diff[1:]) & (diff[:-1] * diff[1:] <= 0)):
        diff_func = interp1d(df['T'], diff, kind='linear')
        Tc = brentq(diff_func, df['T'].min(), df['T'].max())
Tc


In [ ]:
# fig, ax = plt.subplots(dpi=300)

# ax.errorbar(results["T"], results["val_anti"], yerr=results["err_anti"], 
#             fmt='-o', color='black')

# ax.axvline(x=Tc, color='gray', linestyle=':', linewidth=1.5, label=rf'$T_c$')

# ax.set_xlabel(r'$T$')
# ax.set_ylabel(r'$A(\mathbf{k}=\mathbf{M}, \omega=0)$')
# ax.set_xlim(0,0.08) 
# # ax.set_ylim(0.58,0.76) 
# ax.legend(loc='best', frameon=False)
# plt.show()

In [ ]:
fig, ax = plt.subplots(dpi=300)

ax.errorbar(results["T"], results["AN0"], yerr=results["AN0_err"],
            fmt='-o', color='tab:orange', label=r'AN from M-X path')
ax.errorbar(results["T"], results["node0"], yerr=results["node0_err"],
            fmt='-s', color='tab:green', label=r'node from $\Gamma$-X path')
ax.errorbar(results["T"], results["M0"], yerr=results["M0_err"],
            fmt='--^', color='tab:gray', label=r'M point')

if np.isfinite(Tc):
    ax.axvline(x=Tc, color='gray', linestyle=':', linewidth=1.5, label=rf'$T_c={Tc:.4f}$')

ax.set_xlabel(r'$T$')
ax.set_ylabel(r'$A(\omega=0)$')
ax.set_xlim(0, 0.105)
ax.legend(loc='best', frameon=False)
plt.show()


In [ ]:
# ==========================================
# 4. dA_AN(omega=0)/dT
# ==========================================

t_vals = results["T"].copy()
a0_vals = results["AN0"].copy()
mask = np.isfinite(t_vals) & np.isfinite(a0_vals)
t_vals = t_vals[mask]
a0_vals = a0_vals[mask]
sort_idx = np.argsort(t_vals)
t_vals = t_vals[sort_idx]
a0_vals = a0_vals[sort_idx]

if len(t_vals) < 2:
    print("Not enough points to compute dA/dT.")
else:
    dA_dT = np.gradient(a0_vals, t_vals)
    fig, ax = plt.subplots(dpi=300)
    ax.plot(t_vals, dA_dT, '-o', color='blue')
    ax.set_xlabel(r'$T$')
    ax.set_ylabel(r'$dA_{\mathrm{AN}}(0)/dT$')
    plt.show()


In [ ]:
# fig, ax = plt.subplots(dpi=300)

# contrast = (results["peak_XG"] - results["peak_MX"]) / (results["peak_XG"] + results["peak_MX"])
# contrast_err = np.sqrt( (results["err_XG"]**2 + results["err_MX"]**2) ) / (results["peak_XG"] + results["peak_MX"])

# ax.errorbar(results["T"], contrast, yerr=contrast_err, 
#             fmt='-o', color='black')

# ax.axvline(x=Tc, color='gray', linestyle=':', linewidth=1.5, label=rf'$T_c$')
# ax.axhline(y=0, color='black', linestyle='--', linewidth=1)

# ax.set_xlabel(r'$T$')
# ax.set_ylabel('Fermi surface contrast')
# ax.set_xlim(0,0.2) 
# # ax.set_ylim(-0.01,0.21) 
# ax.legend(loc='best', frameon=False)
# plt.show()